In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

B, S = "workspace.bronze", "workspace.silver"

def bronze(name):
    return spark.table(f"{B}.{name}").drop("_ingested_at", "_source_file")

def save(df, name):
    df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{S}.{name}")
    print(f"{name}: {spark.table(f'{S}.{name}').count():,} rows")

In [0]:
ts_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
           "order_delivered_customer_date", "order_estimated_delivery_date"]

orders = bronze("orders").dropDuplicates(["order_id"])
for c in ts_cols:
    orders = orders.withColumn(c, F.to_timestamp(c))

orders = (orders
    .withColumn("delivery_days", F.datediff("order_delivered_customer_date", "order_purchase_timestamp"))
    .withColumn("is_late", F.col("order_delivered_customer_date") > F.col("order_estimated_delivery_date"))
    .withColumn("purchase_month", F.date_trunc("month", "order_purchase_timestamp").cast("date")))
save(orders, "orders")

In [0]:
items = (bronze("order_items")
    .withColumn("order_item_id", F.col("order_item_id").cast("int"))
    .withColumn("price", F.col("price").cast("decimal(10,2)"))
    .withColumn("freight_value", F.col("freight_value").cast("decimal(10,2)"))
    .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date"))
    .dropDuplicates(["order_id", "order_item_id"]))
save(items, "order_items")

payments = (bronze("payments")
    .withColumn("payment_sequential", F.col("payment_sequential").cast("int"))
    .withColumn("payment_installments", F.col("payment_installments").cast("int"))
    .withColumn("payment_value", F.col("payment_value").cast("decimal(10,2)")))
save(payments, "payments")

In [0]:
tr = bronze("category_translation").toDF("product_category_name", "category_en")  # also strips a possible BOM in the header

products = (bronze("products")
    .withColumnRenamed("product_name_lenght", "product_name_length")
    .withColumnRenamed("product_description_lenght", "product_description_length")
    .withColumn("product_weight_g", F.col("product_weight_g").cast("int"))
    .join(tr, "product_category_name", "left")
    .withColumn("category_en", F.coalesce("category_en", "product_category_name", F.lit("unknown")))
    .dropDuplicates(["product_id"]))
save(products, "products")

In [0]:
customers = (bronze("customers")
    .withColumn("customer_city", F.initcap(F.trim("customer_city")))
    .withColumn("customer_state", F.upper(F.trim("customer_state")))
    .dropDuplicates(["customer_id"]))
save(customers, "customers")

sellers = (bronze("sellers")
    .withColumn("seller_city", F.initcap(F.trim("seller_city")))
    .withColumn("seller_state", F.upper(F.trim("seller_state")))
    .dropDuplicates(["seller_id"]))
save(sellers, "sellers")

In [0]:
w = Window.partitionBy("review_id", "order_id").orderBy(F.col("review_answer_timestamp").desc())

reviews = (bronze("reviews")
    .withColumn("review_score", F.col("review_score").cast("int"))
    .withColumn("review_creation_date", F.to_timestamp("review_creation_date"))
    .withColumn("review_answer_timestamp", F.to_timestamp("review_answer_timestamp"))
    .filter(F.col("review_score").between(1, 5))
    .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn"))
save(reviews, "reviews")

In [0]:
def q(sql): return spark.sql(sql).first()[0]

checks = {
    "orders: null order_id": q(f"SELECT COUNT(*) FROM {S}.orders WHERE order_id IS NULL"),
    "items: orphan order_id": q(f"SELECT COUNT(*) FROM {S}.order_items LEFT ANTI JOIN {S}.orders USING (order_id)"),
    "items: negative price": q(f"SELECT COUNT(*) FROM {S}.order_items WHERE price < 0"),
    "products: null product_id": q(f"SELECT COUNT(*) FROM {S}.products WHERE product_id IS NULL"),
}
for name, n in checks.items():
    print("✅" if n == 0 else "❌", name, n)